## 0) Importações e configurações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos (use apenas os que precisar, conforme o desafio escolhido)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Métricas
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error
)

# Para ter resultados reproduzíveis (mude apenas se quiser)
RANDOM_STATE = 42

In [ ]:
# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [14, 8]

## 1) Carregamento dos dados

In [ ]:
# Ajuste o caminho do ficheiro se necessário
DATA_PATH = "volis_dataset.csv"

df = pd.read_csv(DATA_PATH)

## 2) Visão geral do dataset

In [ ]:
print("Dimensão (linhas, colunas):", df.shape)
display(df.sample(5, random_state=RANDOM_STATE))

print("\nTipos de dados:")
print(df.dtypes)

print("\nResumo estatístico (numéricas):")
display(df.describe())

## 3) Qualidade dos dados: missing values e duplicados

In [ ]:
# Missing values por coluna
print("\nValores em falta:")
display(df.isnull().sum())

# Percentual de missing por coluna
print("\nPercentual de valores em falta:")
display(df.isnull().mean() * 100)

# Linhas duplicadas
print("\nLinhas duplicadas:")
print(df.duplicated().sum())

In [ ]:
# Limpeza de dados
# ================

linhas_originais = df.shape[0]
print(f"N. linhas originais: {linhas_originais}")

# Como todos os valores em falta são inferiores a 5%, então vamos remover essas linhas
df_limpo = df.dropna()

linhas_depois = df_limpo.shape[0]
print(f"N. linhas após limpeza: {linhas_depois}")
print(f"Percentual a remover: {(1 - linhas_depois / linhas_originais):.0%}")

print("Percentagem muito elevada. Não remover. Necessário efetuar algumas correções em vez de eliminar linhas.")

In [ ]:
print(f"N. linhas originais: {linhas_originais}")

# Remover linhas cujo target é vazio/NaN
df = df.dropna(subset=["bl_private"])

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

In [ ]:
df['bl_private'] = df['bl_private'].astype(float)

In [ ]:
#Analisar colunas percentagem
colunas_percentagem = ['qt_top_10_percent','qt_top_25_percent','pc_faculty_with_phd','pc_faculty_with_terminal_degree','pc_alumni_donors','pc_graduation_rate']

for col in colunas_percentagem:
    if col in df.columns:
        mask = (df[col] < 0) | (df[col] > 100)
        
        if mask.any():
            print(f"Coluna: {col}")
            display(df[mask])

In [ ]:
#Corrigir percentagem
df['pc_faculty_with_phd'] = df['pc_faculty_with_phd'].replace(103, 100)
#Eliminar linha percentagem
df = df[((df['pc_graduation_rate'] < 118) | (df['pc_graduation_rate'].isnull()))]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

In [ ]:
colunas_numericas = df.columns.tolist()
colunas_numericas.remove('nm_college')

# análise da assimetria
skewness_original = df[colunas_numericas].skew().sort_values(ascending=False)

print("Análise de assimetris (skewness)")
display(skewness_original)

In [ ]:
variaveis_assimetricas = [
    'vl_student_faculty_ratio',
    'qt_postgraduate_students',
    'qt_applications_received',
    'vl_expenditure_per_student',
    'qt_applications_accepted',
    'vl_books_cost',
    'qt_students_enrolled',
    'qt_undergraduate_students',
    'qt_top_10_percent',
    'vl_personal_expenses',
    'pc_alumni_donors'
]

variaveis_simetricas = [
    'vl_tuition_outstate',
    'qt_top_25_percent',
    'vl_room_board',
    'pc_graduation_rate',
    'pc_faculty_with_terminal_degree',
    'pc_faculty_with_phd',
    'bl_private'
]

In [ ]:
# assimetria muito elevada - pearson não serve
corr_spearman = df.corr(method='spearman', numeric_only=True)

# Gráfico: top correlações (ajuste como quiser)

plt.figure()
sns.heatmap(corr_spearman, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

In [ ]:
# Atributos Correlacionados

colunas_graduate = ['qt_applications_received','qt_applications_accepted','qt_students_enrolled','qt_undergraduate_students']
colunas_top_perc = ['qt_top_10_percent','qt_top_25_percent']
colunas_faculty = ['pc_faculty_with_phd','pc_faculty_with_terminal_degree']

In [ ]:
colunas_para_nan = colunas_numericas.copy()
colunas_para_nan.remove('bl_private')

# colunas com valores a 0/1
for col in colunas_para_nan:
    print(f"Coluna: {col}: \n - Total valores inferiores a 1: {df[(df[col] <= 1)][col].count()} \n - Primeiro valor superior a 1 : {df[(df[col] > 1)][col].count()}")

In [ ]:
# Definir valores inferiores ou iguais a 1 como NaN
for col in colunas_para_nan:
    df.loc[(df[col] <= 1), col] = np.nan

In [ ]:
# colunas com outliers mais extremos
outliers_superiores = {}

for col in colunas_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    #lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    max_value = df[col].max()
    #min_value = df[col].min()
    
    distance_upper = max(0, max_value - upper)
    #distance_lower = max(0, lower - min_value)
    
    outliers_superiores[col] = max(distance_upper, Q3)

pd.Series(outliers_superiores).sort_values(ascending=False)

In [ ]:
colunas_outliers_1 = ['vl_expenditure_per_student', 'vl_tuition_outstate']
colunas_outliers_2 = ['qt_applications_received','qt_undergraduate_students','vl_room_board','vl_personal_expenses','qt_applications_accepted','qt_postgraduate_students']
colunas_outliers_3 = ['vl_books_cost', 'qt_students_enrolled']

In [ ]:
plt.figure()
sns.boxplot(data=df[colunas_outliers_1], color='lightblue')

In [ ]:
for col in colunas_outliers_1:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR

    print(f"Coluna: {col}")
    display(df[df[col] > upper].sort_values(by=col,ascending=False).head(10))

In [ ]:
#remover outliers impossíveis
df = df[
        ((df['vl_expenditure_per_student'] < 1000000) | (df['vl_expenditure_per_student'].isnull())) &
        ((df['vl_tuition_outstate'] < 1000000) | (df['vl_tuition_outstate'].isnull()))
    ]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

In [ ]:
# AVALIAR
plt.figure()
sns.boxplot(data=df[colunas_outliers_1], color='lightblue')

In [ ]:
plt.figure()
sns.boxplot(data=df[colunas_outliers_2], color='lightblue')

In [ ]:
for col in colunas_outliers_2:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR

    print(f"Coluna: {col}")
    display(df[df[col] > upper].sort_values(by=col,ascending=False).head(10))

In [ ]:
# Análise em colunas correlacionadas

filtro = df[(df['qt_students_enrolled'] == 500) & 
        (
          (df['qt_applications_accepted'] == 1500) | (df['qt_applications_accepted'].isna())
        )]

filtro_mediana = filtro['qt_applications_received'].median()

print(f"Valor mais comum no filtro: {filtro_mediana}")

In [ ]:
# Uma vez que colunas estão correlacionadas podemos fazer correção por regra lógica

df.loc[(df['qt_applications_received'] == 100000), 'qt_applications_received'] = 2000

In [ ]:
# Aproveitamos para corrigir também os NaN de 'qt_applications_accepted'

filtro = df[(df['qt_applications_received'] == 2000) & (df['qt_students_enrolled'] == 500) & (df['qt_applications_accepted'].notna())]

filtro_mediana = filtro['qt_applications_accepted'].median()

print(f"Valor mais comum no filtro: {filtro_mediana}")

In [ ]:
df.loc[(df['qt_applications_received'] == 20000) & (df['qt_students_enrolled'] == 500) & (df['qt_applications_accepted'].isna()), 'qt_applications_accepted'] = 1500

In [ ]:
#remover outliers impossíveis
df = df[
        ((df['qt_undergraduate_students'] < 50000) | (df['qt_undergraduate_students'].isnull())) &
        ((df['vl_room_board'] < 50000) | (df['vl_room_board'].isnull())) &
        ((df['vl_personal_expenses'] < 30000) | (df['vl_personal_expenses'].isnull()))
    ]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

In [ ]:
# AVALIAR
plt.figure()
sns.boxplot(data=df[colunas_outliers_2], color='lightblue')

In [ ]:
plt.figure()
sns.boxplot(data=df[colunas_outliers_3], color='lightblue')

In [ ]:
for col in colunas_outliers_3:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR

    print(f"Coluna: {col}")
    display(df[df[col] > upper].sort_values(by=col,ascending=False).head(10))

In [ ]:
#remover outliers impossíveis
df = df[
        ((df['vl_books_cost'] < 10000) | (df['vl_books_cost'].isnull()))
    ]

print(f"N. linhas após limpeza: {df.shape[0]}")
print(f"Percentual total de linhas removidas: {(1 - df.shape[0] / linhas_originais):.0%}")

In [ ]:
# AVALIAR
plt.figure()
sns.boxplot(data=df[colunas_outliers_3], color='lightblue')

In [ ]:
# máscara de inconsistência
mask_inconsistente = (
    (df["qt_applications_received"] < df["qt_applications_accepted"]) |
    (df["qt_applications_accepted"] < df["qt_students_enrolled"])
)

df_inconsistentes = df[mask_inconsistente]

print(f"Percentual de linhas inconsistentes: {(df_inconsistentes.shape[0] / df.shape[0]):.2%}")

display(df_inconsistentes[colunas_graduate])

In [ ]:
#Correcao
df.loc[(df['qt_applications_received'] == 2000) & (df['qt_applications_accepted'] == 20000) & (df['qt_students_enrolled'] == 500), 'qt_applications_accepted'] = 2000
df.loc[(df['qt_applications_received'] == 2000) & (df['qt_applications_accepted'] == 1500) & (df['qt_students_enrolled'] == 10000), 'qt_students_enrolled'] = 1000

In [ ]:
# máscara de inconsistência
mask_inconsistente = (df["qt_top_10_percent"] > df["qt_top_25_percent"])

df_inconsistentes = df[mask_inconsistente]

print(f"Percentual de linhas inconsistentes: {(df_inconsistentes.shape[0] / df.shape[0]):.2%}")

display(df_inconsistentes)

In [ ]:
#Correção por regra lógica

df.loc[(df['qt_top_10_percent'] > 0) & (df['qt_top_10_percent'] < 100) & (df['qt_top_10_percent'] > df['qt_top_25_percent']), 'qt_top_25_percent'] = df['qt_top_10_percent']
df.loc[(df['qt_top_25_percent'] > 0) & (df['qt_top_25_percent'] < 100) & (df['qt_top_10_percent'] > df['qt_top_25_percent']), 'qt_top_10_percent'] = df['qt_top_25_percent']

In [ ]:
# máscara de inconsistência
mask_inconsistente = (df["pc_faculty_with_phd"] > df["pc_faculty_with_terminal_degree"])

df_inconsistentes = df[mask_inconsistente]

print(f"Percentual de linhas inconsistentes: {(df_inconsistentes.shape[0] / df.shape[0]):.2%}")

display(df_inconsistentes)

In [ ]:
#Correção por regra lógica

df.loc[(df['pc_faculty_with_phd'] > 0) & (df['pc_faculty_with_phd'] < 100) & (df['pc_faculty_with_phd'] > df['pc_faculty_with_terminal_degree']), 'pc_faculty_with_terminal_degree'] = df['pc_faculty_with_phd']

df.loc[(df['pc_faculty_with_terminal_degree'] > 0) & (df['pc_faculty_with_terminal_degree'] < 100) & (df['pc_faculty_with_phd'] > df['pc_faculty_with_terminal_degree']), 'pc_faculty_with_phd'] = df['pc_faculty_with_terminal_degree']

In [ ]:
print("Variância original:")
variancia_original = (df[colunas_numericas].quantile(0.75) - df[colunas_numericas].quantile(0.25)).sort_values(ascending=False)
print(variancia_original)

print("\nVariáveis correlacionadas")
print(f"Atributos de graduate: {colunas_graduate}")
print(f"Atributos de top percentagem: {colunas_top_perc}")
print(f"Atributos de faculty: {colunas_faculty}")

In [ ]:

## AVALIAR IMPUTACAO POR CORRELAÇÃO COM OUTRAS COLUNAS

print("\n--- Aplicação de Imputação ---")

# Estratégia: Como existem outliers, vamos usar a MEDIANA em vez da Média.
imputer = SimpleImputer(strategy='median').set_output(transform="pandas")

# Nota: O fit_transform retorna um array numpy, precisamos converter de volta para DataFrame
# Aplicamos apenas na coluna 'age' que tinha falhas
df[colunas_numericas] = imputer.fit_transform(df[colunas_numericas])

display(colunas_numericas)

print("Nulos após imputação:")
print(df.isnull().sum())

In [ ]:
# Aplicar transformação logarítmica para suavizar a curva
df_transf = np.log1p(df[colunas_numericas])

In [ ]:
df_scaled = df_transf.copy();

scaler = RobustScaler()
df_scaled[colunas_numericas] = scaler.fit_transform(df_transf[colunas_numericas])

In [ ]:
print("Variância original:")
variancia_original = (df[colunas_numericas].quantile(0.75) - df[colunas_numericas].quantile(0.25)).sort_values(ascending=False)
print(variancia_original)

print("\nVariáveis correlacionadas")
print(f"Atributos de graduate: {colunas_graduate}")
print(f"Atributos de top percentagem: {colunas_top_perc}")
print(f"Atributos de faculty: {colunas_faculty}")

In [ ]:
# Selecionar a coluna com maior variância entre as correlações existentes

# 1. De graduate selecionar 'qt_applications_received'
# 2. De top percentagem selecionar: 'qt_top_25_percent'
# 3. De faculty selecionar: 'pc_faculty_with_terminal_degree'

colunas_para_cluster = [
    'vl_tuition_outstate',
    'vl_expenditure_per_student',
    'qt_applications_received', #'qt_applications_accepted', 'qt_students_enrolled', 'qt_undergraduate_students',
    'qt_top_25_percent', #'qt_top_10_percent',
    #'qt_postgraduate_students',
    'pc_faculty_with_terminal_degree', #'pc_faculty_with_phd'
    'pc_graduation_rate'
]

colunas_para_cluster = [
    'qt_undergraduate_students',
    'qt_applications_received',
    'qt_top_25_percent',
    'pc_faculty_with_phd',
    'pc_graduation_rate',
    'vl_tuition_outstate',
    'vl_expenditure_per_student',
    'vl_room_board',
    'vl_personal_expenses',
    'bl_private'
]

X_scaled = df_scaled[colunas_para_cluster].to_numpy()

In [ ]:
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled) # PCA sempre nos dados escalados!

print(f"O PCA selecionou automaticamente {pca.n_components_} componentes para atingir 95% de variância.")

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca[:,0], X_pca[:,1], X_pca[:,2])
plt.show()

In [ ]:
from sklearn.cluster import KMeans

sse = []
k_range = range(1, 11)

for k in k_range:
    kmeans_elbow = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans_elbow.fit(X_scaled)
    sse.append(kmeans_elbow.inertia_)

# Plotar o gráfico do Cotovelo
plt.figure(figsize=(8, 5))
plt.plot(k_range, sse, marker='o')
plt.title('Método do Cotovelo (Elbow Method)')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inércia (SSE)')
plt.grid(True)
plt.show()

In [ ]:
# Treinando com o k ideal
n_clusters = 3
kmeans_final = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
y_kmeans = kmeans_final.fit_predict(X_pca)

# Visualizando o resultado

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=y_kmeans, s=20, cmap='viridis')
#plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, s=20, cmap='viridis')

# Plotar os centróides
centers = kmeans_final.cluster_centers_
ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2], c='red', s=200, alpha=0.7, label='Centróides')
#ax.title(f"Clusterização Final (k={n_clusters})")
ax.legend()
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_pca, y_kmeans)
print(f"Silhouette Score para k={n_clusters}: {score:.3f}")

# Interpretação rápida
if score > 0.5:
    print("Resultado: Clusters bem separados e definidos.")
elif score > 0.2:
    print("Resultado: Separação razoável.")
else:
    print("Resultado: Clusters sobrepostos ou mal definidos.")

In [ ]:
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 3], c=kmeans_final.labels_, cmap='viridis', alpha=0.6)
# ax.xaxis.label = "PC1"
# ax.yaxis.label = "PC2"
# ax.zaxis.label = "PC3"
# #ax.title("Clusters visualizados nos 3 primeiros componentes")
# #plt.colorbar(label="Cluster")
# plt.show()

In [ ]:
# Adiciona os clusters ao dataframe original

df['cluster'] = y_kmeans

In [ ]:
# Tamanho de cada cluster
# Isso mostra se tens clusters equilibrados ou um grupo dominante

cluster_size = df["cluster"].value_counts().sort_index()
print(cluster_size)

In [ ]:
# Estatísticas globais
media_global = df[variaveis_simetricas].mean()
mediana_global = df[variaveis_assimetricas].median()

estat_global = pd.concat([media_global, mediana_global])

In [ ]:
# Estatísticas por cluster
media_cluster = df[variaveis_simetricas + ['cluster']].groupby('cluster').mean()
mediana_cluster = df[variaveis_assimetricas + ['cluster']].groupby('cluster').median()

estat_cluster = media_cluster.copy()

for col in mediana_cluster.columns:
    estat_cluster[col] = mediana_cluster[col]

In [ ]:
# Comparar com média global

# - Valores positivos → acima da mediana
# - Valores negativos → abaixo da mediana

desvio_assimetricas = df[variaveis_assimetricas].std()
desvio_simetricas = df[variaveis_simetricas].std()
desvio_simetricas['bl_private'] =  np.sqrt(df['bl_private'].mean()*(1-df['bl_private'].mean()))

diff_desvio_simetricas = (media_cluster - media_global) / desvio_simetricas
diff_desvio_assimetricas = (mediana_cluster - mediana_global) / desvio_assimetricas

diff_norm = pd.concat([diff_desvio_simetricas, diff_desvio_assimetricas], axis=1)

In [ ]:
# variáveis que mais diferenciam cada cluster.

# diferença absoluta
#cluster_diff_abs = cluster_diff.abs()

top_features = {}

for cluster in diff_norm.index:
    cluster_effects = diff_norm.loc[cluster]
    top = cluster_effects.sort_values(ascending=False)
    top_features[cluster] = top

display(top_features)

In [ ]:
plt.figure()
#sns.heatmap(corr_spearman, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

#plt.figure(figsize=(12, 6))

sns.heatmap(
    diff_norm,
    annot=True,          # mostra os valores no quadrado
    fmt=".2f",           # formato numérico
    cmap="coolwarm",         # cores crescentes com intensidade
    linewidths=0.5,
    linecolor="gray"
)

plt.title("Heatmap de Intensidade das Variáveis por Cluster", fontsize=16)
plt.ylabel("Cluster", fontsize=12)
plt.xlabel("Variáveis", fontsize=12)
plt.xticks(rotation=45, ha='right')  # gira nomes das colunas
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("cluster")["bl_private"].mean()